# InsureRAG-VLM End-to-End Demo

This notebook demonstrates how to build the retrieval index, query the VLM pipeline, run evaluation, compare clause differences, and extract PDF pages.

## 1. Import Required Libraries

Import the pipeline and helper modules for retrieval, OCR, evaluation, diff analysis, and PDF extraction.

In [ ]:
from pathlib import Path
import os
import sys

# Make imports work whether the notebook is launched from the repo root or notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.insurerag_vlm.config import ModelConfig
from src.insurerag_vlm.pipeline import DocumentRetrievalPipeline
from src.insurerag_vlm.diff import compare_clause_diff, render_clause_diff, summarize_clause_diff
from src.insurerag_vlm.evaluation import generate_evaluation_examples
from src.insurerag_vlm.pdf import extract_text_by_page, extract_layout_by_page
from src.insurerag_vlm.preprocess import PageImagePreprocessConfig, preprocess_page_images

from src.insurerag_vlm.qa import generate_policy_qa_pairs, compute_retrieval_metrics
from src.insurerag_vlm.visual import build_visual_index, compute_visual_retrieval_metrics
from src.insurerag_vlm.ablation import run_ablation
from src.insurerag_vlm.diff import write_policy_diff
print(f'Project root: {project_root}')


## 2. Build the Retrieval Index

Build a text/image/PDF retrieval index from a data folder. Set the proper API keys before running this cell.

## 1.5 Page-Image Preprocessing

Render PDFs into page images and create ColQwen2/ColPali-compatible metadata.


In [ ]:
raw_pdf_dir = project_root / 'data' / '00_raw' / 'public'
preprocess_result = preprocess_page_images(
    PageImagePreprocessConfig(
        input_dir=raw_pdf_dir,
        output_root=project_root / 'data',
        render_dpi=150,
        run_ocr=False,
    )
)
print('Documents:', preprocess_result.document_count)
print('Pages:', preprocess_result.page_count)
print('Page manifest:', preprocess_result.page_manifest_path)
print('Retrieval pairs:', preprocess_result.retrieval_pairs_path)


In [ ]:
data_folder = project_root / 'data' / '00_raw' / 'public'

openai_key = os.getenv('OPENAI_API_KEY')
hf_token = os.getenv('HF_API_TOKEN')
use_openai = bool(openai_key)
use_hf = bool(hf_token) and not use_openai

config = ModelConfig(
    use_hf_api=use_hf,
    retrieval_model=(
        os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
        if use_openai
        else os.getenv('HF_RETRIEVAL_MODEL', ModelConfig().retrieval_model)
    ),
    vlm_model=(
        os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')
        if use_openai
        else os.getenv('HF_VLM_MODEL', ModelConfig().vlm_model)
    ),
    openai_api_key=openai_key,
    hf_api_token=hf_token,
    index_dir=project_root / 'data',
)
pipeline = DocumentRetrievalPipeline(config)
pipeline.build_index(data_folder)
print('Index built successfully:', config.index_path.exists(), config.metadata_path.exists())
print('Data folder:', data_folder)
print('Embedding backend:', 'OpenAI' if use_openai else 'Hugging Face' if use_hf else 'Local hashing baseline')
print('Retrieval model:', config.retrieval_model)
print('Generation model:', config.vlm_model)


## 2.5 QA/Evidence Generation

Generate answerable policy QA pairs plus lexical hard negatives from the loaded PDFs.


In [ ]:
qa_result = generate_policy_qa_pairs(
    data_folder=project_root / 'data' / '00_raw' / 'public',
    output_dir=project_root / 'data' / '02_processed',
)
print('QA pairs:', qa_result.qa_count, qa_result.qa_path)
print('Hard negatives:', qa_result.hard_negative_count, qa_result.hard_negatives_path)


## 3. Query the Pipeline with Page Ranking

Retrieve the top candidate pages and generate a grounded answer from the VLM.

In [ ]:
question = 'What is the comprehensive deductible?'
result = pipeline.query_with_ranking(question, data_folder, top_k=3)
print('Answer:\n', result['answer'])
print('\nTop ranked pages:')
for candidate in result['source_ranking']:
    print(f"- {candidate['source']} (score={candidate['score']:.4f})")


## 3.5 Retrieval Metrics

Evaluate whether retrieved pages match the generated evidence pages.


In [ ]:
retrieval_metrics = compute_retrieval_metrics(
    pipeline,
    data_folder=project_root / 'data' / '00_raw' / 'public',
    qa_path=project_root / 'data' / '02_processed' / 'qa_pairs.jsonl',
    top_k=3,
)
print(retrieval_metrics)


## 4. Evaluate QA Predictions

Run evaluation over a JSON file of question-answer examples to compute EM, F1, and citation precision.

In [ ]:
examples_path = data_folder / 'synthetic_eval_examples.json'
if examples_path.exists():
    metrics = pipeline.evaluate(data_folder, examples_path, top_k=3)
    print('Evaluation metrics:')
    for name, value in metrics.items():
        print(f'{name}: {value:.4f}')
else:
    print('synthetic_eval_examples.json not found; please create a dataset to run evaluation.')


## 4.5 Generate Evaluation Dataset

Convert QA input files into the JSON evaluation format used by the pipeline.


## 5. Clause Diff with Sentence-Level Scoring

Compare two documents and inspect the diff summary and top sentence changes.

In [ ]:
original_path = Path('policy_v1.txt')
revised_path = Path('policy_v2.txt')
if original_path.exists() and revised_path.exists():
    old_text = original_path.read_text(encoding='utf-8', errors='ignore')
    new_text = revised_path.read_text(encoding='utf-8', errors='ignore')
    changes = compare_clause_diff(old_text, new_text)
    print('Clause diff:')
    print(render_clause_diff(changes)[:2000])
    print('\nDiff summary:')
    print(summarize_clause_diff(old_text, new_text))
else:
    print('policy_v1.txt or policy_v2.txt not found; add sample files to compare.')


## 6. PDF Extraction and Layout

Show how to extract PDF text and layout blocks using the built-in helpers.

In [ ]:
pdf_path = data_folder / 'synthetic_auto_policy.pdf'
if pdf_path.exists():
    page_texts = extract_text_by_page(pdf_path)
    print(f'Extracted {len(page_texts)} pages from PDF.')
    print('Page 1 preview:')
    print(page_texts[0][:800])
    layouts = extract_layout_by_page(pdf_path)
    print(f'Page layout blocks on page 1: {len(layouts[0].blocks)}')
else:
    print('synthetic_auto_policy.pdf not found; add a PDF file to run extraction.')


## 6. Visual Retrieval Stub

Build and evaluate the page-image retrieval interface used by future ColPali/ColQwen2 backends.


In [ ]:
visual_index_result = build_visual_index(
    project_root / 'data' / '03_index' / 'colqwen2' / 'page_manifest.jsonl',
    project_root / 'data' / '03_index' / 'colqwen2',
    backend='visual_stub',
)
print(visual_index_result)
visual_metrics = compute_visual_retrieval_metrics(
    project_root / 'data' / '02_processed' / 'qa_pairs.jsonl',
    project_root / 'data' / '03_index' / 'colqwen2',
    backend='visual_stub',
    top_k=3,
)
print(visual_metrics)


## 7. Ablation Summary

Generate retrieval and answer metrics for local text and visual stub backends.


In [ ]:
ablation_outputs = run_ablation(
    data_folder=project_root / 'data' / '00_raw' / 'public',
    qa_path=project_root / 'data' / '02_processed' / 'qa_pairs.jsonl',
    output_dir=project_root / 'reports' / 'ablation',
    index_dir=project_root / 'data',
    visual_index_dir=project_root / 'data' / '03_index' / 'colqwen2',
    top_k=3,
)
print(ablation_outputs)
print((project_root / 'reports' / 'ablation' / 'summary.md').read_text())


## 8. Policy Diff Example

Compare two synthetic policy versions for deductible, coverage, endorsement, exclusion, and duties changes.


In [ ]:
policy_diff = write_policy_diff(
    old_text='\n\n'.join(extract_text_by_page(project_root / 'data' / '00_raw' / 'public' / 'synthetic_auto_policy.pdf')),
    new_text='\n\n'.join(extract_text_by_page(project_root / 'data' / '00_raw' / 'public' / 'synthetic_auto_policy_v2.pdf')),
    output_path=project_root / 'reports' / 'diff' / 'diff_summary.json',
)
policy_diff
